# 05 · Injeção de falhas (gabarito de avaliação)
**CRISP-DM:** Preparação dos Dados (conjunto de avaliação)

Sem histórico de incidentes, a avaliação usa **falhas sintéticas controladas** na semana de teste (01–07/09). As falhas são aplicadas nos **registros por minuto × cliente** e depois reagregadas com as mesmas funções do notebook 02. Assim, atributos por cliente (concentração, abrangência) reagem como numa falha real.

| Tipo | Lado | Baixa | Média | Alta |
|---|---|---|---|---|
| Latência elevada | provedor | ×1,5 | ×2,5 | ×4 |
| Pico de 5xx | provedor | 5% | 15% | 40% das respostas 2xx viram 503 |
| Queda de volume | provedor | mantém 70% | 50% | 20% |
| 4xx de 1 cliente | consumidor | 30% | 60% | 90% das respostas 2xx do cliente viram 400 |
| Rajada de 1 cliente | consumidor | ×2 | ×4 | ×8 o volume do cliente |

- Duração sorteada entre 1, 2, 3, 4 e 6 h; um episódio por bloco de 30 h de cada recurso, sem sobreposição.
- O cliente afetado (tipos de consumidor) é sorteado com probabilidade proporcional ao seu volume no episódio.
- **10 sementes** independentes; em cada uma, as 15 combinações tipo × intensidade são distribuídas em rodízio.
- Janela rotulada com o episódio se ≥ 50% dela estiver dentro dele; cobertura parcial fica fora das métricas.

In [1]:
%run ./00_funcoes.ipynb

00_funcoes carregado · raiz: /home/wsl/gh/mestrado/claude9


In [2]:
brutos = pd.read_parquet(RESULTADOS / "02_brutos_consolidados.parquet")
brutos_teste = brutos[(brutos["ts"] >= TESTE[0]) & (brutos["ts"] < TESTE[1])].reset_index(drop=True)
limites = pd.read_csv(RESULTADOS / "02_limites_recursos.csv", index_col=0, parse_dates=["inicio", "fim"])
lim_teste = limites_teste(limites)
jan_limpo = adicionar_tipo_dia_adaptado(pd.read_parquet(RESULTADOS / f"02_janelas_{JANELA}.parquet"))
print(f"registros no teste: {len(brutos_teste):,}")
lim_teste

registros no teste: 296,541


,inicio,fim
recurso,,
14553746,2026-09-01,2026-09-08 00:00:00
17834257,2026-09-01,2026-09-08 00:00:00
18139260,2026-09-01,2026-09-08 00:00:00
18166630,2026-09-01,2026-09-08 00:00:00
18168196,2026-09-01,2026-09-08 00:00:00
18169917,2026-09-01,2026-09-08 00:00:00
18170306,2026-09-01,2026-09-08 00:00:00
18344302,2026-09-01,2026-09-08 00:00:00
18917905,2026-09-01,2026-09-03 07:00:00


In [3]:
(RESULTADOS / "05_series").mkdir(exist_ok=True)
todos_eps = []
for s in SEMENTES:
    eps = planejar(lim_teste, s)
    js = montar_teste(brutos_teste, jan_limpo, eps, JANELA, lim_teste, s)
    js.to_parquet(RESULTADOS / "05_series" / f"serie_{JANELA}_s{s}.parquet", index=False)
    todos_eps.append(eps.assign(semente=s))
    print(f"semente {s}: {len(eps)} episódios · janelas rotuladas {int((js['episodio'] >= 0).sum())}")
episodios = pd.concat(todos_eps, ignore_index=True)
lim_teste.to_csv(RESULTADOS / "05_limites_teste.csv")
episodios.to_csv(RESULTADOS / "05_episodios.csv", index=False)

semente 0: 44 episódios · janelas rotuladas 138


semente 1: 44 episódios · janelas rotuladas 157


semente 2: 44 episódios · janelas rotuladas 127


semente 3: 44 episódios · janelas rotuladas 160


semente 4: 44 episódios · janelas rotuladas 144


semente 5: 44 episódios · janelas rotuladas 132


semente 6: 44 episódios · janelas rotuladas 170


semente 7: 44 episódios · janelas rotuladas 146


semente 8: 44 episódios · janelas rotuladas 129


semente 9: 44 episódios · janelas rotuladas 128


## Composição dos episódios (10 sementes)

In [4]:
comp = episodios.pivot_table(index="tipo", columns="intensidade", values="episodio", aggfunc="count")[INTENSIDADES]
comp["total"] = comp.sum(axis=1)
comp.index = comp.index.map(NOMES_TIPO)
display(comp)
print("duração (h):", episodios["duracao_h"].value_counts().sort_index().to_dict())
print("episódios por recurso (média por semente):")
(episodios.groupby("recurso").size() / len(SEMENTES)).round(1)

intensidade,baixa,media,alta,total
tipo,,,,
4xx de 1 cliente,29,27,29,85
Pico de 5xx,30,29,30,89
Latência elevada,30,29,29,88
Queda de volume,30,29,30,89
Rajada de 1 cliente,30,29,30,89


duração (h): {1: 83, 2: 83, 3: 92, 4: 93, 6: 89}
episódios por recurso (média por semente):


recurso
14553746    5.0
17834257    5.0
18139260    5.0
18166630    5.0
18168196    5.0
18169917    5.0
18170306    5.0
18344302    5.0
18917905    1.0
18917906    1.0
18917920    1.0
18917921    1.0
dtype: float64

## Verificação: efeito da injeção nos atributos
Comparação, na semente 0, entre as janelas do episódio com e sem a falha (mesma janela na série limpa).

In [5]:
js0 = pd.read_parquet(RESULTADOS / "05_series" / f"serie_{JANELA}_s0.parquet")
eps0 = episodios[episodios["semente"] == 0]
limpo_idx = jan_limpo.set_index(["recurso", "janela"])
linhas = []
for ep in eps0.itertuples():
    w = js0[js0["episodio"] == ep.episodio]
    if w.empty:
        continue
    base = limpo_idx.loc[list(zip(w["recurso"], w["janela"]))]
    linhas.append(dict(tipo=NOMES_TIPO[ep.tipo], intensidade=ep.intensidade,
                       vol=w["vol"].sum() / base["vol"].sum(),
                       lat_2xx=w["lat_2xx"].mean() / base["lat_2xx"].mean(),
                       tx_5xx=w["tx_5xx"].mean() - base["tx_5xx"].mean(),
                       tx_4xx=w["tx_4xx"].mean() - base["tx_4xx"].mean(),
                       top1_vol=w["top1_vol"].mean() - base["top1_vol"].mean()))
efeito = pd.DataFrame(linhas).groupby(["tipo", "intensidade"]).mean().round(3)
efeito.columns = ["volume (razão)", "latência 2xx (razão)", "Δ taxa 5xx", "Δ taxa 4xx", "Δ part. maior cliente"]
efeito

volume (razão)  latência 2xx (razão)  Δ taxa 5xx  Δ taxa 4xx  Δ part. maior cliente
tipo                intensidade                                                                                     
4xx de 1 cliente    alta                  1.000                 1.017       0.000       0.184                  0.000
                    baixa                 1.000                 1.000       0.000       0.064                  0.000
                    media                 1.000                 0.996       0.000       0.316                  0.000
Latência elevada    alta                  1.000                 4.000       0.000       0.000                  0.000
                    baixa                 1.000                 1.500       0.000       0.000                  0.000
                    media                 1.000                 2.500       0.000       0.000                  0.000
Pico de 5xx         alta                  1.000                 1.002       0.150       0.000                  0.000
                    baixa                 1.000                 1.001       0.046       0.000                  0.000
                    media                 1.000                 0.998       0.084       0.000                  0.000
Queda de volume     alta                  0.194                 1.012      -0.000      -0.001                  0.008
                    baixa                 0.700                 1.002       0.000      -0.003                  0.003
                    media                 0.508                 0.977      -0.000       0.006                  0.003
Rajada de 1 cliente alta                  5.101                 1.033      -0.000      -0.000                  0.045
                    baixa                 1.840                 1.000      -0.000       0.000                  0.061
                    media                 2.708                 0.998      -0.000      -0.000                  0.059